### oxjob #1991 backfill — stage 1 ("mint or leave blank") over the remaining candidate works, outside the End 2 End
One-off copy of `notebooks/end2end/MatchAuthors` (walden main d15a1f5d) Steps 1-6 and its assignment log, restricted to #1991's candidate works. Its staging tables are `openalex.transient.oxjob1991_bf_*`, so the nightly's (`author_matching_batch`, `pending_author_assignments`, mint tables, `author_rematch_worklist`) are never touched; the nightly's monitoring cells and its log DELETE are left out.

**Run rules** (oxjob #1991): never while the End 2 End's Author_Matching can run (new ids come from MAX(authors.id)); finish before 18:00 CT (the next nightly deletes the assignment log for its UTC date); `dry_run` defaults to TRUE: every production write is `WHERE NOT dry_run`. Writes when not dry: `openalex.authors.authors` (new profiles), `openalex.works.work_authors` (MERGE, as the nightly), `openalex.authors.author_assignment_log` (origin 'oxjob1991_backfill'), `openalex.authors.author_rematch_applied` (so the nightly never re-admits these works).

In [ ]:
-- Run parameters: edit and commit before a run (the commit is the audit trail).
DECLARE OR REPLACE VARIABLE dry_run BOOLEAN DEFAULT TRUE;
DECLARE OR REPLACE VARIABLE n_works INT DEFAULT 1000000;  -- all remaining candidate works (~329K)
SELECT dry_run, n_works;

In [ ]:
-- The next n_works #1991 candidate works not yet admitted (same filter as MatchAuthors cell [7]), in work_id order.
CREATE OR REPLACE TABLE openalex.transient.oxjob1991_bf_worklist AS
SELECT work_id, array('orcid_blank_seat', 'oxjob1991_backfill') AS change_classes, true AS eligible, est_seats,
       current_timestamp() AS admitted_at
FROM (
    SELECT work_id, est_seats, ROW_NUMBER() OVER (ORDER BY work_id) AS rn
    FROM (
        SELECT c.work_id, CAST(COUNT(*) AS INT) AS est_seats
        FROM openalex.transient.oxjob1991_mint_candidates c
        WHERE c.fit = 'match'
          AND NOT EXISTS (SELECT 1 FROM openalex.authors.author_rematch_applied ap
                          WHERE ap.work_id = c.work_id AND ap.run_date > DATE '2026-11-05')
        GROUP BY c.work_id
    )
)
WHERE rn <= system.session.n_works;

SELECT COUNT(*) AS works, SUM(est_seats) AS est_seats FROM openalex.transient.oxjob1991_bf_worklist;

### Step 1: Get updated works that need matching

In [ ]:
-- STEP 1: Create Staging Table — only records that need author matching
-- Resolves institution IDs from work_authors.raw_affiliation_strings (written by affiliations step)
CREATE OR REPLACE TABLE openalex.transient.oxjob1991_bf_batch AS
WITH raw_exploded AS (
    SELECT 
        id AS work_id,
        created_date,
        POSEXPLODE(authorships) AS (author_sequence, authorship)
    FROM openalex.works.openalex_works_base
    WHERE id IN (SELECT work_id FROM openalex.transient.oxjob1991_bf_worklist)  -- backfill: its works only, no watermark
      AND authorships IS NOT NULL 
      AND SIZE(authorships) > 0
),
-- Only keep authorships that need matching:
-- 1. No author_id assigned yet
-- 2. Meets the ID and date cutoffs for new-era matching
needs_matching AS (
    SELECT r.work_id, r.author_sequence, r.authorship,
           r.authorship.raw_author_name AS raw_author_name,
           wa.raw_affiliation_strings
    FROM raw_exploded r
    INNER JOIN openalex.works.work_authors wa
        ON r.work_id = wa.work_id AND r.author_sequence = wa.author_sequence
    WHERE wa.author_id IS NULL
      AND ((r.work_id > 7000000000 AND r.created_date >= to_timestamp('2025-12-20'))
           OR r.work_id IN (SELECT work_id FROM openalex.transient.oxjob1991_bf_worklist))
      -- oxjob #1881: a placeholder name ('Anonymous', 'et al.', '(:unav)', '12292787', ...) is not a person: the seat keeps
      -- its raw name and no author id. Only a seat that carries an ORCID enters (Step 5 lets it attach through the ORCID only).
      AND (openalex.authors.placeholder_author_name(r.authorship.raw_author_name) IS NULL
           OR NULLIF(TRIM(r.authorship.raw_orcid), '') IS NOT NULL)
      -- oxjob #1991 seat gate: for orcid_blank_seat works, only ORCID-carrying seats enter matching
      AND (NOT EXISTS (SELECT 1 FROM openalex.transient.oxjob1991_bf_worklist wl
                       WHERE wl.work_id = r.work_id
                         AND ARRAY_CONTAINS(wl.change_classes, 'orcid_blank_seat'))
           OR NULLIF(TRIM(r.authorship.raw_orcid), '') IS NOT NULL)
),

-- Resolve institution IDs from work_authors.raw_affiliation_strings
exploded_ras AS (
    SELECT nm.work_id, nm.author_sequence, nm.authorship, nm.raw_author_name,
           t.raw_affiliation_string
    FROM needs_matching nm
    LATERAL VIEW OUTER EXPLODE(nm.raw_affiliation_strings) t AS raw_affiliation_string
),
resolved_direct_ids AS (
    SELECT 
        e.work_id, e.author_sequence, e.authorship, e.raw_author_name,
        CASE 
            WHEN e.raw_affiliation_string IS NULL THEN NULL
            WHEN asl.institution_ids IS NOT NULL AND SIZE(asl.institution_ids) > 0 
                THEN asl.institution_ids
            ELSE NULL
        END AS direct_ids
    FROM exploded_ras e
    LEFT JOIN openalex.institutions.raw_affiliation_strings_institutions_mv asl
        ON e.raw_affiliation_string = asl.raw_affiliation_string
),

-- Expand Lineage (FOR MATCHING ONLY)
expanded_for_matching AS (
    SELECT 
        r.work_id,
        r.author_sequence,
        ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(
            FLATTEN(ARRAY(
                FILTER(ARRAY(r.inst_id_scalar), x -> x IS NOT NULL),
                COALESCE(anc.ancestors, ARRAY())
            ))
        ))) as matching_institution_ids
    FROM (
        SELECT work_id, author_sequence, EXPLODE_OUTER(direct_ids) as inst_id_scalar
        FROM resolved_direct_ids
    ) r
    LEFT JOIN (
        SELECT institution_id, lineage_ids as ancestors
        FROM openalex.institutions.institution_ancestors
    ) anc ON CAST(r.inst_id_scalar AS BIGINT) = anc.institution_id
    GROUP BY r.work_id, r.author_sequence
)

SELECT 
    nm.work_id,
    nm.author_sequence,
    COALESCE(efm.matching_institution_ids, ARRAY()) as all_institution_ids,
    nm.authorship as authorship_struct,
    nm.raw_author_name
FROM needs_matching nm
LEFT JOIN expanded_for_matching efm
    ON nm.work_id = efm.work_id AND nm.author_sequence = efm.author_sequence;

### Step 2: Run Matching Algorithm Over Updated Works with ID over 7000000000

In [ ]:
CREATE OR REPLACE TABLE openalex.transient.oxjob1991_bf_pending AS
WITH 
-- 1. ENRICH BATCH DATA
-- Add Signals (Topics, Sources) and parsed name columns to batch data
enriched_batch AS (
  SELECT
    b.work_id,
    b.author_sequence,
    b.raw_author_name,
    b.all_institution_ids,
    TRANSFORM(b.all_institution_ids, x -> CONCAT('https://openalex.org/I', CAST(x AS STRING))) AS institution_ids,
    
    pn.parsed_name.first AS pn_first,
    SUBSTRING(pn.parsed_name.first, 1, 1) AS pn_first_initial,
    pn.parsed_name.middle AS pn_middle,
    pn.parsed_name.last AS pn_last,
    pn.alt_parses AS alt_parses,   -- oxjob #1341
    
    COALESCE(wtf.topics, ARRAY()) AS topics,
    
    ARRAY_DISTINCT(
      TRANSFORM(
        FILTER(w.locations, x -> x.source.id IS NOT NULL),
        x -> x.source.id
      )
    ) AS work_source_ids,
    b.authorship_struct.raw_orcid AS incoming_orcid,
    -- oxjob #1360: seat of a work the #649 worklist re-admitted tonight (log origin 'rematch')
    (rw.work_id IS NOT NULL) AS is_rematch
    
  FROM openalex.transient.oxjob1991_bf_batch b
  LEFT JOIN (SELECT DISTINCT work_id FROM openalex.transient.oxjob1991_bf_worklist) rw
    ON b.work_id = rw.work_id
  LEFT JOIN openalex.authors.author_names pn 
    ON TRIM(b.raw_author_name) = pn.raw_author_name
  LEFT JOIN openalex.works.work_topics_v2 wtf   -- oxjob #1531; [] for not-classifiable works = no topic signal, as a missing row was
    ON b.work_id = wtf.work_id
  -- works_base, NOT openalex_works: openalex_works is rebuilt AFTER this task
  -- in end2end, so joining it misses every new work (work_source_ids empty).
  LEFT JOIN openalex.works.openalex_works_base w 
    ON b.work_id = w.id
),

-- 2. PREPARE MATCHING INPUTS
-- Calculate Block Keys and ID arrays
authors_prepared AS (
  SELECT
    work_id,
    author_sequence,
    raw_author_name,
    pn_first,
    pn_first_initial,
    pn_middle,
    pn_last,
    alt_parses,
    -- Block Key Generation (parsed_name fields are already normalized)
    CASE
      WHEN pn_last IS NULL THEN NULL
      WHEN pn_first_initial IS NULL OR pn_first_initial = '' THEN pn_last
      ELSE CONCAT(pn_first_initial, ' ', pn_last)
    END AS block_key,
    institution_ids,
    -- Extract Topic IDs
    TRANSFORM(topics, t -> t.id) AS topic_ids,
    work_source_ids,
    usable_orcid,
    -- Seats carrying the same name and signals get the same decision; only one representative
    -- per tuple goes through the block join (a 20K-seat org roster otherwise lands on one task).
    -- is_rematch splits rematch seats into their own tuples: they are held to a different variant bar.
    is_rematch,
    xxhash64(raw_author_name, institution_ids, TRANSFORM(topics, t -> t.id), work_source_ids, usable_orcid, is_rematch) AS tuple_key
  FROM (
    SELECT *,
      -- publishers sometimes stamp one author's ORCID on every authorship of a work;
      -- an ORCID on >1 authorship of the same work is untrustworthy (name cascade decides)
      CASE WHEN COUNT(*) OVER (PARTITION BY work_id, incoming_orcid) = 1 THEN incoming_orcid END AS usable_orcid
    FROM enriched_batch
    WHERE raw_author_name IS NOT NULL
  )
),

signal_reps AS (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY tuple_key ORDER BY work_id, author_sequence) AS rn
    FROM authors_prepared
  ) WHERE rn = 1
),

-- 2b. ORCID MATCHING (global — no block constraint)
-- Match incoming ORCID to any profile holding it. If several profiles share
-- the ORCID (splinters), take the most-cited, then most works, then oldest id.
orcid_matches AS (
  SELECT
    ap.work_id,
    ap.author_sequence,
    COUNT(DISTINCT a.author_id) AS orcid_match_count,
    MAX_BY(a.author_id, STRUCT(COALESCE(a.cited_by_count, 0), COALESCE(a.works_count, 0), -a.author_id)) AS orcid_author_id
  FROM signal_reps ap
  -- any trusted ORCID the profile has ever carried resolves to it, not just the primary (#1267);
  -- exploded so the join stays an equi-join
  JOIN (
    SELECT author_id, cited_by_count, works_count, EXPLODE(observed_orcids) AS orcid
    FROM openalex.authors.authors_for_matching
    WHERE observed_orcids IS NOT NULL AND SIZE(observed_orcids) > 0
    UNION ALL
    -- Profiles not yet in authors_for_matching: minted since its 12:21 UTC rebuild (this run can
    -- finish before or after that), or zero-works / unparseable-name. Read the mint table directly
    -- so a seat carrying an ORCID we already minted cannot mint a twin (oxjob #444; the stale-lookup
    -- nights of 2026-07-15/16 minted 59K same-ORCID profiles). Zero stats: they lose every tiebreak
    -- to an established holder. authors.orcid is the mint-time value, so honour curated removals.
    SELECT a.id AS author_id, 0 AS cited_by_count, 0 AS works_count, a.orcid
    FROM openalex.authors.authors a
    LEFT ANTI JOIN openalex.authors.authors_for_matching f ON f.author_id = a.id
    LEFT JOIN openalex.authors.author_orcid_curations oc ON oc.author_id = a.id
    WHERE a.orcid IS NOT NULL AND a.orcid <> '' AND NOT (a.orcid <=> oc.removed_orcid)
  ) a
    ON a.orcid = ap.usable_orcid
  GROUP BY ap.work_id, ap.author_sequence
),

-- 3. CANDIDATE BLOCKING
-- 3a. Primary block (unchanged): the seat's block key equals the profile's block key.
primary_candidates AS (
  SELECT
    e.tuple_key,
    FALSE AS via_variant,
    alm.author_id,
    alm.display_name,
    alm.first,
    alm.first_initial,
    alm.middle,
    alm.last,
    alm.institution_ids,
    alm.topic_ids,
    alm.source_ids,
    alm.works_count,
    CAST(NULL AS STRING) AS v_pn_first,
    CAST(NULL AS STRING) AS v_pn_last,
    CAST(NULL AS STRING) AS v_kind
  FROM signal_reps e
  JOIN openalex.authors.authors_for_matching alm
    ON alm.block_key = e.block_key
    AND e.block_key != ''
),

-- 3b. Variant reach (oxjob #1341): any key the seat carries (primary + alternates) that a profile
--     carries (primary + alternates, capped at 10K-profile blocks in authors_for_matching_keys),
--     minus the pairs the primary block already produced. Compound surnames, reversed order and
--     transliteration variants land here. These candidates are held to a stricter bar (see eligible).
seat_keys_raw AS (
  SELECT tuple_key, block_key AS key, TRUE AS is_primary, pn_first AS s_first, pn_last AS s_last, pn_first AS p_first, pn_last AS p_last
  FROM signal_reps
  WHERE block_key IS NOT NULL AND block_key != ''
  UNION ALL
  SELECT tuple_key, p.key, FALSE, p.first AS s_first, p.last AS s_last, pn_first AS p_first, pn_last AS p_last
  FROM signal_reps
  LATERAL VIEW explode(COALESCE(alt_parses, ARRAY())) AS p
),

-- Reading kinds (oxjob #1341), inferred from how a reading's (first, last) relate to the side's primary parse:
--   primary; fold ('f:' keys); reversed (first/last swapped); reversed_middle (primary surname as given
--   name, a middle token as surname); surname_token (first kept, last is a token or join/split form of the primary surname);
--   middle (first kept, last is a middle token or middle + surname); other (romanizations, ALL-CAPS surname).
-- Opus 5.5 judged 935 variant matches of the 2026-09-24 batch: reversed_middle 15% precise, the pairs
-- surname_token|surname_token 31%, reversed|middle 35%, primary(seat)|middle(profile) 58%, middle|middle 76%;
-- everything else 81-95%. Dropping those lifts precision 72% -> 91% and keeps 58% of the cross-block pairs.
seat_keys AS (
  SELECT *,
    CASE WHEN is_primary THEN 'primary'
         WHEN key LIKE 'f:%' THEN 'fold'
         WHEN s_first = p_last AND s_last = p_first THEN 'reversed'
         WHEN s_first = p_last THEN 'reversed_middle'
         WHEN s_first = p_first AND (p_last = s_last OR array_contains(split(p_last, '[ -]'), s_last)
              OR regexp_replace(p_last, '[ -]', '') = regexp_replace(s_last, '[ -]', '') OR endswith(p_last, CONCAT(' ', s_last))) THEN 'surname_token'
         WHEN s_first = p_first THEN 'middle'
         ELSE 'other' END AS s_kind
  FROM seat_keys_raw
),

profile_keys AS (
  SELECT *,
    CASE WHEN is_primary THEN 'primary'
         WHEN key LIKE 'f:%' THEN 'fold'
         WHEN first = p_last AND last = p_first THEN 'reversed'
         WHEN first = p_last THEN 'reversed_middle'
         WHEN first = p_first AND (p_last = last OR array_contains(split(p_last, '[ -]'), last)
              OR regexp_replace(p_last, '[ -]', '') = regexp_replace(last, '[ -]', '') OR endswith(p_last, CONCAT(' ', last))) THEN 'surname_token'
         WHEN first = p_first THEN 'middle'
         ELSE 'other' END AS k_kind
  FROM openalex.authors.authors_for_matching_keys
),

-- one row per (seat tuple, profile): the first shared key by key order, with both sides' readings
-- under that key, so names are compared under the alternate reading (a reversed reading swaps
-- first/last; a surname-token reading uses the token as surname on both sides).
variant_ids AS (
  SELECT
    sk.tuple_key,
    fk.author_id,
    MIN_BY(sk.s_first, sk.key) AS v_pn_first,
    MIN_BY(sk.s_last,  sk.key) AS v_pn_last,
    MIN_BY(fk.first,   sk.key) AS v_cand_first,
    MIN_BY(fk.last,    sk.key) AS v_cand_last,
    MIN_BY(CONCAT(sk.s_kind, '|', fk.k_kind), sk.key) AS v_kind
  FROM seat_keys sk
  JOIN profile_keys fk
    ON fk.key = sk.key
    AND NOT (sk.is_primary AND fk.is_primary)
    -- allowed reading pairs only (see the kinds comment above)
    AND sk.s_kind <> 'reversed_middle' AND fk.k_kind <> 'reversed_middle'
    AND NOT (sk.s_kind = 'surname_token' AND fk.k_kind = 'surname_token')
    AND NOT (sk.s_kind = 'reversed' AND fk.k_kind = 'middle') AND NOT (sk.s_kind = 'middle' AND fk.k_kind = 'reversed')
    AND NOT (sk.s_kind = 'primary' AND fk.k_kind = 'middle')
    AND NOT (sk.s_kind = 'middle' AND fk.k_kind = 'middle')
  LEFT ANTI JOIN primary_candidates pc
    ON pc.tuple_key = sk.tuple_key AND pc.author_id = fk.author_id
  GROUP BY sk.tuple_key, fk.author_id
),

variant_candidates AS (
  SELECT
    vi.tuple_key,
    TRUE AS via_variant,
    alm.author_id,
    alm.display_name,
    vi.v_cand_first AS first,
    SUBSTRING(vi.v_cand_first, 1, 1) AS first_initial,
    CAST(NULL AS STRING) AS middle,
    vi.v_cand_last AS last,
    alm.institution_ids,
    alm.topic_ids,
    alm.source_ids,
    alm.works_count,
    vi.v_pn_first,
    vi.v_pn_last,
    vi.v_kind
  FROM variant_ids vi
  JOIN openalex.authors.authors_for_matching alm
    ON alm.author_id = vi.author_id
),

all_candidates AS (
  SELECT * FROM primary_candidates
  UNION ALL
  SELECT * FROM variant_candidates
),

blocked_candidates AS (
  SELECT 
    e.tuple_key,
    e.work_id,
    e.author_sequence,
    e.raw_author_name,
    e.pn_first,
    e.pn_first_initial,
    e.pn_middle,
    e.pn_last,
    e.block_key,
    e.institution_ids,
    e.topic_ids,
    e.work_source_ids,
    e.is_rematch,
    COALESCE(c.via_variant, FALSE) AS via_variant,
    c.v_kind,
    -- name-comparison columns (oxjob #1341): the seat's primary parse, or its alternate reading for a
    -- variant-reached candidate. pn_* stay the primary parse (they key the seat's group below).
    COALESCE(c.v_pn_first, e.pn_first) AS cmp_first,
    SUBSTRING(COALESCE(c.v_pn_first, e.pn_first), 1, 1) AS cmp_first_initial,
    CASE WHEN COALESCE(c.via_variant, FALSE) THEN NULL ELSE e.pn_middle END AS cmp_middle,
    COALESCE(c.v_pn_last, e.pn_last) AS cmp_last,
    c.author_id,
    c.display_name AS candidate_display_name,
    c.first AS cand_first,
    c.first_initial AS cand_first_initial,
    c.middle AS cand_middle,
    c.last AS cand_last,
    c.institution_ids as candidate_institution_ids,
    c.topic_ids as candidate_topic_ids,
    c.source_ids AS candidate_source_ids,
    c.works_count
  FROM signal_reps e
  LEFT JOIN all_candidates c
    ON c.tuple_key = e.tuple_key
),

with_match_signals AS (
  SELECT
    *,
    NAMED_STRUCT(
      'id', author_id,
      'display_name', candidate_display_name,
      'v_kind', v_kind,
      'has_topic', (size(topic_ids) > 0 AND size(candidate_topic_ids) > 0 AND arrays_overlap(candidate_topic_ids, topic_ids))
    ) AS candidate_obj,
    
    (size(institution_ids) > 0 AND size(candidate_institution_ids) > 0 
     AND arrays_overlap(candidate_institution_ids, institution_ids)) as has_inst,
    
    (size(topic_ids) > 0 AND size(candidate_topic_ids) > 0 
     AND arrays_overlap(candidate_topic_ids, topic_ids)) as has_topic,

     (SIZE(work_source_ids) > 0 AND SIZE(candidate_source_ids) > 0
     AND ARRAYS_OVERLAP(candidate_source_ids, work_source_ids)) AS has_source
  FROM blocked_candidates
),

-- oxjob #1341: a candidate reached only through an alternate key must also share an institution and
-- a source or topic with the seat, and hold at most 20 works; otherwise it is not a candidate at all.
-- Primary-block candidates are eligible unconditionally (unchanged behaviour). The works cap comes from
-- Opus 5.5 judging 742 variant matches of the 2026-09-24 batch: profiles <= 20 works 96.4% precise,
-- 21-100 works 84%, > 100 works ~50% (large common-name profiles are conflated clusters); a shared
-- topic did not help (reversed readings with a shared topic were 76%).
-- oxjob #1360: rematch seats (#649 worklist) never match through a variant key. Opus 5.5 on the 2026-09-26
-- nightly: variant matches of rematch seats 63% same person (52/82), of new works 89% (59/66).
with_eligibility AS (
  SELECT
    *,
    (author_id IS NOT NULL AND (NOT via_variant OR (NOT is_rematch AND has_inst AND (has_source OR has_topic) AND COALESCE(works_count, 0) <= 20))) AS eligible
  FROM with_match_signals
),

with_name_matches AS (
  SELECT
    *,
    -- 1: Exact Full Name (both have full first, full middle, same last)
    (LENGTH(cmp_first) > 1 AND LENGTH(cmp_middle) > 1 AND LENGTH(cand_first) > 1 AND LENGTH(cand_middle) > 1
     AND cmp_first = cand_first
     AND cmp_middle = cand_middle
     AND cmp_last = cand_last
    ) AND eligible as pattern_1_exact_full,

    -- 2: Exact First, Middle Initial match (batch has full first + middle initial only)
    (LENGTH(cmp_first) > 1 AND (cmp_middle IS NULL OR LENGTH(cmp_middle) <= 1)
     AND LENGTH(cand_first) > 1
     AND cmp_first = cand_first
     AND cmp_last = cand_last
     AND (cand_middle IS NULL OR cmp_middle IS NULL OR SUBSTRING(cmp_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1))
    ) AND eligible as pattern_2_exact_first_mid_init,

    -- 3: Initials match to Full (batch has first initial + middle, candidate has full)
    (LENGTH(cmp_first) = 1 AND cmp_middle IS NOT NULL
     AND LENGTH(cand_first) > 1 AND cand_middle IS NOT NULL
     AND cmp_first_initial = cand_first_initial
     AND SUBSTRING(cmp_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1)
     AND cmp_last = cand_last
    ) AND eligible as pattern_3_init_mid_init,

    -- 4: First Initial, Middle Initial match (both have only initials, no full names)
    (LENGTH(cmp_first) = 1 AND LENGTH(cand_first) = 1
     AND cmp_middle IS NOT NULL AND cand_middle IS NOT NULL
     AND LENGTH(cmp_middle) <= 1 AND LENGTH(cand_middle) <= 1
     AND cmp_first_initial = cand_first_initial
     AND SUBSTRING(cmp_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1)
     AND cmp_last = cand_last
    ) AND eligible as pattern_4_first_init_mid_init,

    -- 5: Exact First, Exact Last (no middle)
    (LENGTH(cmp_first) > 1 AND LENGTH(cand_first) > 1
     AND cmp_first = cand_first
     AND cmp_last = cand_last
     AND cmp_middle IS NULL
    ) AND eligible as pattern_5_exact_first_last,

    -- 6: First Initial Only to Full (batch has first initial only, candidate has full first)
    (LENGTH(cmp_first) = 1 AND cmp_middle IS NULL
     AND LENGTH(cand_first) > 1
     AND cmp_first_initial = cand_first_initial
     AND cmp_last = cand_last
    ) AND eligible as pattern_6_first_init_to_full,

    -- 7: First Initial Only (both have only first initial, no middle)
    (LENGTH(cmp_first) = 1 AND LENGTH(cand_first) = 1
     AND cmp_middle IS NULL AND cand_middle IS NULL
     AND cmp_first_initial = cand_first_initial
     AND cmp_last = cand_last
    ) AND eligible as pattern_7_first_init_last,

    -- 8: Full Name to Initial (batch has full first, candidate has only initial)
    (LENGTH(cmp_first) > 1 AND LENGTH(cand_first) = 1
     AND cmp_first_initial = cand_first_initial
     AND cmp_last = cand_last
    ) AND eligible as pattern_8_full_to_init

  FROM with_eligibility
),

with_any_name_match AS (
  SELECT
    *,
    (pattern_1_exact_full OR pattern_2_exact_first_mid_init OR pattern_3_init_mid_init OR 
     pattern_4_first_init_mid_init OR pattern_5_exact_first_last OR pattern_6_first_init_to_full OR 
     pattern_7_first_init_last OR pattern_8_full_to_init) as any_name_match
  FROM with_name_matches
),

aggregated_counts AS (
  SELECT
    tuple_key,
    work_id,
    author_sequence,
    raw_author_name,
    block_key,
    institution_ids,
    pn_first,
    pn_first_initial,
    pn_middle,
    pn_last,
    work_source_ids,
    
    -- STRATEGY 1: Name Only (Unique)
    count_if(pattern_1_exact_full AND NOT via_variant) AS s1_n1, count_if(pattern_2_exact_first_mid_init AND NOT via_variant) AS s1_n2,
    count_if(pattern_3_init_mid_init AND NOT via_variant) AS s1_n3, count_if(pattern_4_first_init_mid_init AND NOT via_variant) AS s1_n4,
    count_if(pattern_5_exact_first_last AND NOT via_variant) AS s1_n5, count_if(pattern_6_first_init_to_full AND NOT via_variant) AS s1_n6,
    count_if(pattern_7_first_init_last AND NOT via_variant) AS s1_n7, count_if(pattern_8_full_to_init AND NOT via_variant) AS s1_n8,
    
    -- STRATEGY 2: Name + Institution
    count_if(pattern_1_exact_full AND has_inst AND NOT via_variant) AS s2_n1, count_if(pattern_2_exact_first_mid_init AND has_inst AND NOT via_variant) AS s2_n2,
    count_if(pattern_3_init_mid_init AND has_inst AND NOT via_variant) AS s2_n3, count_if(pattern_4_first_init_mid_init AND has_inst AND NOT via_variant) AS s2_n4,
    count_if(pattern_5_exact_first_last AND has_inst AND NOT via_variant) AS s2_n5, count_if(pattern_6_first_init_to_full AND has_inst AND NOT via_variant) AS s2_n6,

    -- STRATEGY 6: Name + Inst + Source
    count_if(pattern_1_exact_full AND has_inst AND has_source AND NOT via_variant) AS s6_n1,
    count_if(pattern_2_exact_first_mid_init AND has_inst AND has_source AND NOT via_variant) AS s6_n2,
    count_if(pattern_5_exact_first_last AND has_inst AND has_source AND NOT via_variant) AS s6_n5,
    count_if(pattern_6_first_init_to_full AND has_inst AND has_source AND NOT via_variant) AS s6_n6,
    count_if(pattern_7_first_init_last AND has_inst AND has_source AND NOT via_variant) AS s6_n7,

    -- STRATEGY 4: Name + Inst + Topic
    count_if(pattern_1_exact_full AND has_inst AND has_topic AND NOT via_variant) AS s4_n1,
    count_if(pattern_2_exact_first_mid_init AND has_inst AND has_topic AND NOT via_variant) AS s4_n2,
    count_if(pattern_5_exact_first_last AND has_inst AND has_topic AND NOT via_variant) AS s4_n5,
    count_if(pattern_6_first_init_to_full AND has_inst AND has_topic AND NOT via_variant) AS s4_n6,
    count_if(pattern_7_first_init_last AND has_inst AND has_topic AND NOT via_variant) AS s4_n7,

    -- STRATEGY 5: Name + Source
    count_if(pattern_1_exact_full AND has_source AND NOT via_variant) AS s5_n1,
    count_if(pattern_2_exact_first_mid_init AND has_source AND NOT via_variant) AS s5_n2,
    count_if(pattern_5_exact_first_last AND has_source AND NOT via_variant) AS s5_n5,
    count_if(pattern_6_first_init_to_full AND has_source AND NOT via_variant) AS s5_n6,
    count_if(pattern_7_first_init_last AND has_source AND NOT via_variant) AS s5_n7,

    -- STRATEGY 3: Name + Topic
    count_if(pattern_1_exact_full AND has_topic AND NOT via_variant) AS s3_n1,
    count_if(pattern_2_exact_first_mid_init AND has_topic AND NOT via_variant) AS s3_n2,
    count_if(pattern_5_exact_first_last AND has_topic AND NOT via_variant) AS s3_n5,
    
    -- CAPTURE MATCHED OBJECTS
    MAX(CASE WHEN pattern_1_exact_full AND NOT via_variant THEN candidate_obj END) AS match_s1_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND NOT via_variant THEN candidate_obj END) AS match_s1_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND NOT via_variant THEN candidate_obj END) AS match_s1_n5,

    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n6,
    
    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s3_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s3_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s3_n5,
    
    -- VARIANT TIERS (oxjob #1341): candidates reached only through an alternate key. Counted apart
    -- from the tiers above so they can never displace or tie a primary-block match; they fire only when
    -- the primary cascade found nothing. Full first name under the alternate reading, plus institution
    -- and a source (v6) or a topic (v4); eligibility already required inst + (source | topic).
    count_if(pattern_1_exact_full AND via_variant AND has_inst AND has_source) AS v6_n1,
    count_if(pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_source) AS v6_n2,
    count_if(pattern_5_exact_first_last AND via_variant AND has_inst AND has_source) AS v6_n5,
    count_if(pattern_1_exact_full AND via_variant AND has_inst AND has_topic) AS v4_n1,
    count_if(pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_topic) AS v4_n2,
    count_if(pattern_5_exact_first_last AND via_variant AND has_inst AND has_topic) AS v4_n5,
    MAX(CASE WHEN pattern_1_exact_full AND via_variant AND has_inst AND has_source THEN candidate_obj END) AS match_v6_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_source THEN candidate_obj END) AS match_v6_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND via_variant AND has_inst AND has_source THEN candidate_obj END) AS match_v6_n5,
    MAX(CASE WHEN pattern_1_exact_full AND via_variant AND has_inst AND has_topic THEN candidate_obj END) AS match_v4_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_topic THEN candidate_obj END) AS match_v4_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND via_variant AND has_inst AND has_topic THEN candidate_obj END) AS match_v4_n5,

    COUNT_IF(author_id IS NOT NULL AND NOT via_variant) AS total_candidates_in_block,   -- primary block only, as before
    COUNT_IF(any_name_match AND NOT via_variant) AS total_name_matches

  FROM with_any_name_match
  GROUP BY tuple_key, work_id, author_sequence, raw_author_name, block_key, institution_ids,
           pn_first, pn_first_initial, pn_middle, pn_last, work_source_ids
),

final_decision AS (
SELECT
  ac.tuple_key,
  ac.work_id,
  ac.author_sequence,
  ac.block_key,
  raw_author_name,
  institution_ids,
  pn_first,
  pn_first_initial,
  pn_last,
  work_source_ids,
  
  -- MATCH OUTCOME (all n8 tiers retired, oxjob #691: 11-63% judge-measured precision)
  CASE 
    WHEN om.orcid_author_id IS NOT NULL THEN 'MATCHED'
    WHEN (
      s1_n1=1 OR s1_n2=1 OR s1_n5=1 OR 
      s6_n1=1 OR s6_n2=1 OR s6_n5=1 OR s6_n6=1 OR
      s2_n1=1 OR s2_n2=1 OR s2_n5=1 OR s2_n6=1 OR
      s4_n1=1 OR s4_n2=1 OR s4_n5=1 OR s4_n6=1 OR
      s5_n1=1 OR s5_n2=1 OR s5_n5=1 OR s5_n6=1 OR
      s3_n1=1 OR s3_n2=1 OR s3_n5=1 OR
      v6_n1=1 OR v6_n2=1 OR v6_n5=1
    ) THEN 'MATCHED'
    WHEN total_candidates_in_block = 0 THEN 'NO_CANDIDATES'
    ELSE 'AMBIGUOUS'
  END AS match_outcome,

  -- NAME-BASED AUTHOR ID (existing name-pattern cascade)
  CASE 
    WHEN s1_n1 = 1 THEN match_s1_n1.id
    WHEN s1_n2 = 1 THEN match_s1_n2.id
    WHEN s1_n5 = 1 THEN match_s1_n5.id
    
    WHEN s6_n1 = 1 THEN match_s6_n1.id
    WHEN s6_n2 = 1 THEN match_s6_n2.id
    WHEN s6_n5 = 1 THEN match_s6_n5.id
    WHEN s6_n6 = 1 THEN match_s6_n6.id

    WHEN s2_n1 = 1 THEN match_s2_n1.id
    WHEN s2_n2 = 1 THEN match_s2_n2.id
    WHEN s2_n5 = 1 THEN match_s2_n5.id
    WHEN s2_n6 = 1 THEN match_s2_n6.id

    WHEN s4_n1 = 1 THEN match_s4_n1.id
    WHEN s4_n2 = 1 THEN match_s4_n2.id
    WHEN s4_n5 = 1 THEN match_s4_n5.id
    WHEN s4_n6 = 1 THEN match_s4_n6.id

    WHEN s5_n1 = 1 THEN match_s5_n1.id
    WHEN s5_n2 = 1 THEN match_s5_n2.id
    WHEN s5_n5 = 1 THEN match_s5_n5.id
    WHEN s5_n6 = 1 THEN match_s5_n6.id

    WHEN s3_n1 = 1 THEN match_s3_n1.id
    WHEN s3_n2 = 1 THEN match_s3_n2.id
    WHEN s3_n5 = 1 THEN match_s3_n5.id

    -- oxjob #1341: variant tiers, only when nothing above fired. v4 (inst + topic) is counted but not
    -- used: Opus 5.5 judged 8 of 14 such matches right on the 2026-09-23 batch (reversed common pinyin
    -- names with a shared topic), v6 (inst + source) 42 of 45.
    WHEN v6_n1 = 1 THEN match_v6_n1.id
    WHEN v6_n2 = 1 THEN match_v6_n2.id
    WHEN v6_n5 = 1 THEN match_v6_n5.id

    ELSE NULL
  END AS name_author_id,

  -- WHICH name tier fired (observational, oxjob #640): same WHEN order as
  -- name_author_id above, so this names the tier that produced that id.
  -- Consumed by AuthorshipDailyMetrics; no matching logic reads it.
  CASE 
    WHEN s1_n1 = 1 THEN 's1_n1'
    WHEN s1_n2 = 1 THEN 's1_n2'
    WHEN s1_n5 = 1 THEN 's1_n5'

    WHEN s6_n1 = 1 THEN 's6_n1'
    WHEN s6_n2 = 1 THEN 's6_n2'
    WHEN s6_n5 = 1 THEN 's6_n5'
    WHEN s6_n6 = 1 THEN 's6_n6'

    WHEN s2_n1 = 1 THEN 's2_n1'
    WHEN s2_n2 = 1 THEN 's2_n2'
    WHEN s2_n5 = 1 THEN 's2_n5'
    WHEN s2_n6 = 1 THEN 's2_n6'

    WHEN s4_n1 = 1 THEN 's4_n1'
    WHEN s4_n2 = 1 THEN 's4_n2'
    WHEN s4_n5 = 1 THEN 's4_n5'
    WHEN s4_n6 = 1 THEN 's4_n6'

    WHEN s5_n1 = 1 THEN 's5_n1'
    WHEN s5_n2 = 1 THEN 's5_n2'
    WHEN s5_n5 = 1 THEN 's5_n5'
    WHEN s5_n6 = 1 THEN 's5_n6'

    WHEN s3_n1 = 1 THEN 's3_n1'
    WHEN s3_n2 = 1 THEN 's3_n2'
    WHEN s3_n5 = 1 THEN 's3_n5'

    WHEN v6_n1 = 1 THEN 'v6_n1'
    WHEN v6_n2 = 1 THEN 'v6_n2'
    WHEN v6_n5 = 1 THEN 'v6_n5'
    ELSE NULL
  END AS name_match_tier,

  -- oxjob #1341: reading pair and topic overlap of the chosen variant match (observability)
  CASE WHEN v6_n1 = 1 THEN match_v6_n1.v_kind WHEN v6_n2 = 1 THEN match_v6_n2.v_kind WHEN v6_n5 = 1 THEN match_v6_n5.v_kind END AS via_kind,
  CASE WHEN v6_n1 = 1 THEN match_v6_n1.has_topic WHEN v6_n2 = 1 THEN match_v6_n2.has_topic WHEN v6_n5 = 1 THEN match_v6_n5.has_topic END AS via_has_topic,

  -- ORCID tier (global, most-cited holder). orcid_match_count = GLOBAL number
  -- of profiles holding the ORCID (0 = no holder or no usable ORCID).
  COALESCE(om.orcid_match_count, 0) AS orcid_match_count,
  om.orcid_author_id

FROM aggregated_counts ac
LEFT JOIN orcid_matches om
  ON ac.work_id = om.work_id
  AND ac.author_sequence = om.author_sequence
)
SELECT
  ap.work_id,
  ap.author_sequence,
  fd.block_key,
  ap.raw_author_name,
  fd.institution_ids,
  fd.pn_first,
  fd.pn_first_initial,
  fd.pn_last,
  fd.work_source_ids,
  -- oxjob #1348: vector decision (small + medium blocks, non-ORCID seats): band 0.93 / 0.95 + hub-pref, or Claude Sonnet 5.5 (high confidence) on the top-3 (oxjob #1401); it replaces the
  -- name cascade's pick for the seats it decided; ORCID still wins below; seats without a decision keep the cascade's outcome.
  CASE WHEN vd.vec_author_id IS NOT NULL AND fd.orcid_author_id IS NULL THEN 'MATCHED' ELSE fd.match_outcome END AS match_outcome,
  COALESCE(vd.vec_author_id, fd.name_author_id) AS name_author_id,
  CASE WHEN vd.vec_author_id IS NOT NULL THEN CONCAT('vec_', vd.rule) ELSE fd.name_match_tier END AS name_match_tier,
  -- oxjob #1341: TRUE when the final assignment came from a variant tier (alternate block key)
  (fd.orcid_author_id IS NULL AND fd.name_author_id IS NOT NULL AND fd.name_match_tier LIKE 'v%') AS via_variant,
  fd.via_kind,
  fd.via_has_topic,
  fd.orcid_match_count,
  fd.orcid_author_id,
  -- FINAL AUTHOR ID: ORCID wins over the vector decision, which wins over the name cascade (oxjob #1348)
  COALESCE(fd.orcid_author_id, vd.vec_author_id, fd.name_author_id) AS existing_author_id,
  CASE 
    WHEN fd.orcid_author_id IS NOT NULL THEN 'orcid'
    WHEN vd.vec_author_id IS NOT NULL THEN 'vector'
    WHEN fd.name_author_id IS NOT NULL THEN 'name'
    ELSE NULL
  END AS match_method,
  -- QA: ORCID picked a different author than the name cascade would have
  (fd.orcid_author_id IS NOT NULL AND COALESCE(vd.vec_author_id, fd.name_author_id) IS NOT NULL
   AND fd.orcid_author_id <> COALESCE(vd.vec_author_id, fd.name_author_id)) AS orcid_name_conflict,
  -- QA: ORCID match with zero name corroboration (name cascade found nothing).
  -- This is the population to sample for wrong-ORCID publisher metadata.
  (fd.orcid_author_id IS NOT NULL AND COALESCE(vd.vec_author_id, fd.name_author_id) IS NULL) AS orcid_blind_match
FROM authors_prepared ap
JOIN final_decision fd
  ON fd.tuple_key = ap.tuple_key AND fd.raw_author_name = ap.raw_author_name
-- oxjob #1348: per-seat vector decision for tonight (band / sonnet rows carry vec_author_id; zone_unresolved / mint / mega rows do not)
LEFT JOIN openalex.authors.vector_match_decisions vd
  ON vd.run_date = current_date() AND vd.work_id = ap.work_id AND vd.author_sequence = ap.author_sequence AND vd.vec_author_id IS NOT NULL;

### Step 3: Cluster Unmatched & Mint New IDs

In [ ]:
-- A. Get the current High Water Mark
DECLARE OR REPLACE VARIABLE max_id BIGINT;
-- High-water mark includes deleted profiles (openalex.authors.deleted_authors, oxjob #1354):
-- deleting the newest empty profiles must never let their ids be re-issued.
SET VARIABLE max_id = (
  SELECT GREATEST(
    (SELECT MAX(id) FROM openalex.authors.authors),
    (SELECT COALESCE(MAX(author_id), 0) FROM openalex.authors.deleted_authors)
  )
);

-- B. Seat -> cluster (oxjob #444). Persisted so Step 5 and the mint-provenance metric read the same
--    key instead of recomputing it. A seat carrying a USABLE ORCID clusters on the ORCID (one person,
--    two works in one batch, different affiliations used to mint two same-ORCID profiles); everything
--    else clusters on name + (institutions | sources) as before.
CREATE OR REPLACE TABLE openalex.transient.oxjob1991_bf_mint_seats AS
WITH orcid_use AS (
    -- same guard as the ORCID tier: an ORCID on >1 authorship of one work is publisher-stamped
    SELECT work_id, author_sequence, raw_orcid,
           CASE WHEN COUNT(*) OVER (PARTITION BY work_id, raw_orcid) = 1 THEN raw_orcid END AS usable_orcid
    FROM (SELECT work_id, author_sequence, NULLIF(TRIM(authorship_struct.raw_orcid), '') AS raw_orcid
          FROM openalex.transient.oxjob1991_bf_batch)
    WHERE raw_orcid IS NOT NULL
)
SELECT
    pa.work_id,
    pa.author_sequence,
    pa.raw_author_name,
    pa.match_outcome,
    ou.raw_orcid,
    ou.usable_orcid,
    CASE
        WHEN ou.usable_orcid IS NOT NULL THEN xxhash64('orcid', ou.usable_orcid)
        ELSE xxhash64(
            -- 1. NAME PART: Normalized if available, else Raw
            CASE
                WHEN pa.pn_first IS NOT NULL AND pa.pn_first != '' AND pa.pn_last IS NOT NULL
                THEN CONCAT(pa.pn_first, ' ', pa.pn_last)
                WHEN pa.pn_first_initial IS NOT NULL AND pa.pn_first_initial != '' AND pa.pn_last IS NOT NULL
                THEN CONCAT(pa.pn_first_initial, ' ', pa.pn_last)
                ELSE LOWER(TRIM(pa.raw_author_name))
            END,
            -- 2. SIGNAL PART: Institutions -> Sources
            CASE
                WHEN SIZE(b.all_institution_ids) > 0
                THEN concat_ws('|', sort_array(b.all_institution_ids))
                ELSE concat_ws('|', sort_array(pa.work_source_ids))
            END
        )
    END AS cluster_hash
FROM openalex.transient.oxjob1991_bf_pending pa
--  Join Batch to get 'all_institution_ids'
LEFT JOIN openalex.transient.oxjob1991_bf_batch b
    ON pa.work_id = b.work_id AND pa.author_sequence = b.author_sequence
LEFT JOIN orcid_use ou
    ON pa.work_id = ou.work_id AND pa.author_sequence = ou.author_sequence
LEFT JOIN openalex.works.work_authors existing
    ON pa.work_id = existing.work_id
    AND pa.author_sequence = existing.author_sequence
WHERE
    -- Only unmatched records
    pa.match_outcome <> 'MATCHED'
    -- ensure we haven't already assigned an ID in a previous run
    AND existing.author_id IS NULL
    -- Safety: Ensure we actually have a name string to hash
    AND pa.raw_author_name IS NOT NULL
    AND TRIM(pa.raw_author_name) <> ''
    -- backfill only: mint only #1991's candidate seats (the judged population). Other ORCID seats on these works
    -- (949 in the 14 Nov full dry run, on 222 giant-byline works) usually print an ORCID a profile already held: blank.
    AND EXISTS (SELECT 1 FROM openalex.transient.oxjob1991_mint_candidates mc
                WHERE mc.work_id = pa.work_id AND mc.author_sequence = pa.author_sequence)
    -- oxjob #1881: never mint a profile for a placeholder name
    AND openalex.authors.placeholder_author_name(pa.raw_author_name) IS NULL
    -- oxjob #1991 same-name guard: for orcid_blank_seat seats, skip the mint when a same-name
    -- profile already exists (ORCID record given+family vs display_name, exact case-folded match).
    -- Prevents duplicate profiles; seats with no ORCID record or no name in it still mint.
-- usable_orcid is a URL, r.orcid a bare 19-character id: compare the last 19 characters (an equality,
-- so a hash join; LIKE CONCAT('%', r.orcid) ran 3 h+ on a nightly batch, oxjob #1642 / #1991).
    AND NOT (
        ou.usable_orcid IS NOT NULL
        AND EXISTS (SELECT 1 FROM openalex.transient.oxjob1991_bf_worklist wl
                    WHERE wl.work_id = pa.work_id
                      AND ARRAY_CONTAINS(wl.change_classes, 'orcid_blank_seat'))
        AND EXISTS (
            SELECT 1
            FROM openalex.authors.oxjob1374_orcid_records r
            JOIN openalex.authors.openalex_authors a
                ON lower(a.display_name) = lower(CONCAT(COALESCE(r.given, ''), ' ', COALESCE(r.family, '')))
               AND a.works_count > 0
            WHERE upper(r.orcid) = upper(RIGHT(ou.usable_orcid, 19))
              AND r.given IS NOT NULL AND r.family IS NOT NULL
        )
    );

-- C0. oxjob #1944: the name each catalogue string holds (openalex.authors.clean_byline_name, #1912), for naming the
--     profiles minted below. Its own table: the warehouse allows 5 Python UDF calls per query, and every use of the
--     column would count as one. The RLIKE gate is CreateWorkAuthorships' fixes-cell gate (a superset of the function's).
CREATE OR REPLACE TABLE openalex.transient.oxjob1991_bf_mint_clean_names AS
SELECT raw, openalex.authors.clean_byline_name(raw) AS clean_name
FROM (
  SELECT DISTINCT raw_author_name AS raw FROM openalex.transient.oxjob1991_bf_mint_seats
  WHERE raw_author_name RLIKE '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]|(?i)(https?://|www\\.|id\\.loc\\.gov|viaf\\.org/|orcid\\.org/|lattes\\.cnpq\\.br/|id_orcid)|(^|\\s)880-[0-9]{2}|[0-9]{3,4}\\??[\\s\\x{00A0}]*-[\\s\\x{00A0}]*([0-9]{3,4}\\??)?[\\s\\x{00A0}]*[.,)]?[\\s\\x{00A0}]*$|[0-9]{3,4}\\??\\s*-\\s*([0-9]{3,4}\\??)?\\s+\\p{L}|\\S[\\s\\x{00A0}]{3,}\\S|[0-9]{4}- ?[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]|\\.\\s*(auteur|éditeur|traducteur|illustrateur|préfacier|compositeur|librettiste|graveur|photographe|cartographe|imprimeur|libraire|dessinateur|annotateur|commentateur|rédacteur)'
);

-- C. Cluster and Mint
-- oxjob #1944: a profile minted from a library-catalogue string is named with the name the string holds, not the
-- first-comma flip of the string ('Mazet, Georges. Auteur du texte' was minted 'Georges. Auteur du texte Mazet', and
-- the profile page and every byline showed it). match_name keeps the flip: it becomes full_name, the matching name, so
-- blocking and matching are unchanged. The clean name is used only when the cleaner changed more than whitespace, it
-- holds no digit, names one person (no ';') and shares a word with the raw string as the rematch (cell 3) compares
-- them, so the rematch can never flag the profile's own seat. Revert: display_name back to flip_name (and cell 14's
-- full_name back to display_name); drop the C0 statement.
CREATE OR REPLACE TABLE openalex.transient.oxjob1991_bf_new_author_queue AS
WITH unique_clusters AS (
    SELECT
        cluster_hash,
        MAX_BY(raw_author_name, length(raw_author_name)) as raw_display_name,
        -- guarded: the publisher-stamped copy of a coauthor's ORCID never lands on a new profile (#444)
        MAX(usable_orcid) as orcid,
        monotonically_increasing_id() as batch_row_id
    FROM openalex.transient.oxjob1991_bf_mint_seats
    GROUP BY cluster_hash
),
named AS (
    SELECT
        uc.*,
        CASE
            WHEN SIZE(SPLIT(uc.raw_display_name, ',')) = 2 THEN
                TRIM(SPLIT(uc.raw_display_name, ',')[1]) || ' ' || TRIM(SPLIT(uc.raw_display_name, ',')[0])
            ELSE
                uc.raw_display_name
        END AS flip_name,
        REGEXP_REPLACE(TRIM(cn.clean_name), '\\s+', ' ') AS clean_name
    FROM unique_clusters uc
    LEFT JOIN openalex.transient.oxjob1991_bf_mint_clean_names cn ON cn.raw = uc.raw_display_name
)
SELECT
    cluster_hash,
    CASE
        WHEN clean_name IS NOT NULL
         AND clean_name <> REGEXP_REPLACE(TRIM(raw_display_name), '\\s+', ' ')
         AND clean_name NOT RLIKE '[0-9]' AND clean_name RLIKE '\\p{L}' AND LENGTH(clean_name) >= 2
         AND INSTR(clean_name, chr(59)) = 0
         AND ARRAYS_OVERLAP(FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(clean_name,'^([^,]+),(.*)$',concat(chr(36), '2 ', chr(36), '1')),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t -> LENGTH(t) >= 3), FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(raw_display_name,'^([^,]+),(.*)$',concat(chr(36), '2 ', chr(36), '1')),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t -> LENGTH(t) >= 3))
        THEN clean_name
        ELSE flip_name
    END AS display_name,
    flip_name AS match_name,
    orcid,
    max_id + ROW_NUMBER() OVER (ORDER BY batch_row_id) AS new_author_id
FROM named;

In [ ]:
-- review: today's match rates (display only; the rows above are the durable copy)
SELECT 
    pa.match_outcome, 
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM openalex.transient.oxjob1991_bf_pending pa
GROUP BY pa.match_outcome

UNION ALL

-- 2. MINTING STATS
SELECT 
    'NEW_AUTHORS_TO_CREATE' as match_outcome,
    COUNT(*) as count,
    NULL as percentage
FROM openalex.transient.oxjob1991_bf_new_author_queue;

### Step 4: Write New Authors to Profiles

In [ ]:
INSERT INTO openalex.authors.authors 
    (id, display_name, full_name, orcid, created_date, updated_date)
SELECT 
    new_author_id AS id,
    display_name,
    match_name AS full_name,  -- the first-comma flip, as before (oxjob #1944 renames only display_name)
    orcid,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.transient.oxjob1991_bf_new_author_queue
WHERE NOT dry_run;

### Step 5: Consolidate Decisions

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW batch_author_decisions AS
SELECT
    b.work_id,
    b.author_sequence,
    b.raw_author_name,

    -- oxjob #1881: placeholder attaches only through its ORCID, never by name/vector/mint
    -- oxjob #1991: for orcid_blank_seat works, keep mints only — leave binds blank
    CASE WHEN openalex.authors.placeholder_author_name(b.raw_author_name) IS NOT NULL THEN pa.orcid_author_id
         WHEN wl_1991.work_id IS NOT NULL THEN q.new_author_id
         ELSE COALESCE(pa.existing_author_id, q.new_author_id) END AS final_author_id

FROM openalex.transient.oxjob1991_bf_batch b

LEFT JOIN openalex.transient.oxjob1991_bf_pending pa
    ON b.work_id = pa.work_id
    AND b.author_sequence = pa.author_sequence

-- seat -> cluster key persisted in Step 3 (only unmatched seats are in it), oxjob #444
LEFT JOIN openalex.transient.oxjob1991_bf_mint_seats ms
    ON b.work_id = ms.work_id
    AND b.author_sequence = ms.author_sequence

LEFT JOIN openalex.transient.oxjob1991_bf_new_author_queue q
    ON q.cluster_hash = ms.cluster_hash

-- oxjob #1991: mint-only gate for orcid_blank_seat class
LEFT JOIN (SELECT DISTINCT work_id FROM openalex.transient.oxjob1991_bf_worklist
           WHERE ARRAY_CONTAINS(change_classes, 'orcid_blank_seat')) wl_1991
    ON b.work_id = wl_1991.work_id;

### Step 6: Update work_authors — author_id only

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (SELECT * FROM batch_author_decisions WHERE NOT dry_run) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence

WHEN MATCHED THEN
    UPDATE SET
        target.author_id = source.final_author_id,
        target.raw_author_name = source.raw_author_name,
        target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (work_id, author_sequence, author_id, raw_author_name, created_at, updated_at)
    VALUES (source.work_id, source.author_sequence, source.final_author_id,
            source.raw_author_name, current_timestamp(), current_timestamp())

### Assignment log (origin 'oxjob1991_backfill'; no DELETE: the nightly owns today's partition)

In [ ]:
-- origin: 'rematch' = admitted by the #649 worklist; 'new_work' = work created for this run;
-- 'restamped' = an older work whose updated_date bump pulled its unbound seats back in.
INSERT INTO openalex.authors.author_assignment_log
  (run_date, work_id, author_sequence, raw_author_name, block_key, block_size, match_outcome, match_method,
   name_match_tier, existing_author_id, orcid_author_id, orcid_match_count, institution_ids, work_source_ids,
   primary_source_id, primary_source_name, provenance, work_created_date, origin, logged_at, via_variant)
WITH block_sizes AS (
    SELECT block_key, COUNT(*) AS block_size
    FROM openalex.authors.authors_for_matching
    WHERE block_key IN (SELECT DISTINCT block_key FROM openalex.transient.oxjob1991_bf_pending WHERE block_key IS NOT NULL)
    GROUP BY block_key
),
worklist AS (
    SELECT DISTINCT work_id FROM openalex.transient.oxjob1991_bf_worklist
)
SELECT current_date(),
       p.work_id,
       CAST(p.author_sequence AS INT),
       p.raw_author_name,
       p.block_key,
       CAST(COALESCE(b.block_size, 0) AS INT),
       p.match_outcome,
       p.match_method,
       p.name_match_tier,
       p.existing_author_id,
       p.orcid_author_id,
       CAST(p.orcid_match_count AS INT),
       CAST(mb.all_institution_ids AS ARRAY<STRING>),
       CAST(p.work_source_ids AS ARRAY<STRING>),
       w.primary_location.source.id,
       w.primary_location.source.display_name,
       w.primary_location.provenance,
       w.created_date,
       'oxjob1991_backfill',
       current_timestamp(),
       COALESCE(p.via_variant, FALSE)
FROM openalex.transient.oxjob1991_bf_pending p
LEFT JOIN block_sizes b ON p.block_key = b.block_key
LEFT JOIN openalex.transient.oxjob1991_bf_batch mb ON p.work_id = mb.work_id AND p.author_sequence = mb.author_sequence
LEFT JOIN openalex.works.openalex_works_base w ON p.work_id = w.id
LEFT JOIN worklist wl ON p.work_id = wl.work_id
WHERE NOT dry_run

In [ ]:
-- Mark the works admitted, so the nightly's cell [7] never re-admits them.
INSERT INTO openalex.authors.author_rematch_applied
SELECT current_date(), work_id, change_classes, eligible, est_seats, admitted_at
FROM openalex.transient.oxjob1991_bf_worklist
WHERE NOT dry_run;

In [ ]:
-- Run summary (stage 1's pass conditions: binds written = 0 on candidate seats).
SELECT dry_run,
  (SELECT COUNT(*) FROM openalex.transient.oxjob1991_bf_worklist) AS works,
  (SELECT COUNT(*) FROM openalex.transient.oxjob1991_bf_batch) AS seats_in_batch,
  (SELECT COUNT_IF(match_outcome = 'MATCHED') FROM openalex.transient.oxjob1991_bf_pending) AS matched_left_blank,
  (SELECT COUNT(*) FROM openalex.transient.oxjob1991_bf_mint_seats) AS mint_seats,
  (SELECT COUNT(*) FROM openalex.transient.oxjob1991_bf_new_author_queue) AS new_profiles,
  (SELECT COUNT_IF(final_author_id IS NOT NULL AND final_author_id NOT IN (SELECT new_author_id FROM openalex.transient.oxjob1991_bf_new_author_queue))
     FROM batch_author_decisions) AS binds_decided;